In [1]:
!pip install kafka-python

In [2]:
import json
import pandas as pd
from kafka import KafkaProducer

In [3]:
# -----------------------------
# Kafka Producer (batch-friendly)
# -----------------------------
producer = KafkaProducer(
bootstrap_servers="kafka:29092",
value_serializer=lambda v: json.dumps(v).encode("utf-8"),
linger_ms=5000 # batch messages before sending
)

In [6]:
df = pd.read_csv('/home/jovyan/data/olist_order_items_dataset.csv')
print("Rows to publish:", len(df))

Rows to publish: 112650


In [7]:
df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [8]:
# ==============================
# CEK DATA SEBELUM TRANSFORMASI
# ==============================

# 1. Ukuran data
print("=== Shape ===")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

# 2. Tipe data
print("\n=== Data Types ===")
print(df.dtypes)

# 3. Jumlah null per kolom
print("\n=== Null Count ===")
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_summary = pd.DataFrame({"null_count": null_counts, "null_%": null_pct})
print(null_summary[null_summary["null_count"] > 0])  # hanya tampilkan yang ada null

# 4. Duplikat
print("\n=== Duplicates ===")
print(f"Duplicate rows     : {df.duplicated().sum():,}")
print(f"Duplicate order_id : {df['order_id'].duplicated().sum():,}")

# 5. Preview data
print("=== Preview (head 3) ===")
df.head(3)

=== Shape ===
Rows: 112,650 | Columns: 7

=== Data Types ===
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

=== Null Count ===
Empty DataFrame
Columns: [null_count, null_%]
Index: []

=== Duplicates ===
Duplicate rows     : 0
Duplicate order_id : 13,984
=== Preview (head 3) ===


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


In [9]:
# -----------------------------
# Transformasi tipe data
# -----------------------------

# Kolom datetime (yang tidak boleh null)
df["shipping_limit_date"] = pd.to_datetime(df["shipping_limit_date"])

# Kolom datetime yang boleh null
nullable_datetime_cols = [
    "shipping_limit_date"
]
for col in nullable_datetime_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")  # NaN tetap NaN, tidak error

# Verifikasi hasil
print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())

order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object

Null counts:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64


In [ ]:
# -----------------------------
# Publish to Kafka in batch
# -----------------------------
def safe_datetime(val):
    """Konversi datetime ke ISO string, return None jika NaT/null"""
    return val.isoformat() if pd.notna(val) else None

for _, row in df.iterrows():
    msg = {
        "order_id": row["order_id"],
        "order_item_id": row["order_item_id"],
        "product_id": row["product_id"],
        "seller_id": row["seller_id"],
        "shipping_limit_date": safe_datetime(row["shipping_limit_date"]),
        "price": float(row["price"]),
        "freight_value": float(row["freight_value"]),
    }
    producer.send("orders_item", msg)

producer.flush()
print("✅Batch publish to Kafka finished")